In [1]:
!pip install gradio -q

In [1]:
import numpy as np
import cv2
import gradio as gr
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. DEFINISI ARSITEKTUR MODEL (CNN-CBAM) - SUDAH DIPERBAIKI UNTUK KERAS 3
# =====================================================================

def channel_attention(input_feature, ratio=8):
    """Channel Attention Module (CAM)"""
    channel = input_feature.shape[-1]

    shared_layer_one = layers.Dense(channel // ratio, activation='relu', kernel_initializer='he_normal', use_bias=True, bias_initializer='zeros')
    shared_layer_two = layers.Dense(channel, kernel_initializer='he_normal', use_bias=True, bias_initializer='zeros')

    # AvgPool
    avg_pool = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool = shared_layer_one(avg_pool)
    avg_pool = shared_layer_two(avg_pool)

    # MaxPool
    max_pool = layers.GlobalMaxPooling2D()(input_feature)
    max_pool = shared_layer_one(max_pool)
    max_pool = shared_layer_two(max_pool)

    cbam_feature = layers.Add()([avg_pool, max_pool])
    cbam_feature = layers.Activation('sigmoid')(cbam_feature)

    return layers.Multiply()([input_feature, cbam_feature])

def spatial_attention(input_feature):
    """Spatial Attention Module (SAM) - FIX: Menggunakan layers.Lambda"""
    # Menggunakan layers.Lambda agar operasi native TensorFlow aman dari error KerasTensor
    avg_pool = layers.Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(input_feature)
    max_pool = layers.Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(input_feature)

    concat = layers.Concatenate(axis=-1)([avg_pool, max_pool])
    cbam_feature = layers.Conv2D(filters=1, kernel_size=7, padding='same', activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(concat)

    return layers.Multiply()([input_feature, cbam_feature])

def cbam_block(cbam_feature):
    """Integrasi CAM dan SAM"""
    cbam_feature = channel_attention(cbam_feature)
    cbam_feature = spatial_attention(cbam_feature)
    return cbam_feature

def build_cnn_cbam_extractor(input_shape=(224, 224, 3)):
    """Membangun Deep Feature Extractor CNN-CBAM"""
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, (3, 3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = cbam_block(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2 - 4 (Ekstraksi fitur hierarkis)
    for filters in [64, 64, 64]:
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = cbam_block(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D((2, 2))(x)

    # Block 5 - 6
    for filters in [128, 128]:
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = cbam_block(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D((2, 2))(x)

    # Flatten untuk menghasilkan 1152-dimensional feature vector
    outputs = layers.Flatten()(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

# Inisialisasi Model Extractor
feature_extractor = build_cnn_cbam_extractor()

# =====================================================================
# 2. SIMULASI TRAINING DATASET KECIL (SMALL-SAMPLE DATASET)
# =====================================================================
print("Menginisialisasi simulasi pelatihan model hybrid CNN-CBAM-SVM...")
np.random.seed(42)

# Membuat dummy data gambar (224, 224, 3) sebanyak 40 sampel (20 murni, 20 palsu)
dummy_images = np.random.rand(40, 224, 224, 3).astype(np.float32)
dummy_labels = np.array([1]*20 + [0]*20)

# Langkah A: Ekstraksi fitur visual grafik via CNN-CBAM
extracted_features = feature_extractor.predict(dummy_images)

# Langkah B: Normalisasi fitur menggunakan StandardScaler
scaler = StandardScaler()
scaled_features = scaler.fit_transform(extracted_features)

# Langkah C: Melatih Pembatas Margin Maksimal (SVM Linear)
svm_classifier = SVC(kernel='linear', probability=True, C=1.0)
svm_classifier.fit(scaled_features, dummy_labels)
print("Model hybrid CNN-CBAM-SVM berhasil dibuat dan siap digunakan!\n")

# =====================================================================
# 3. FUNGSI PIPELINE PREDIKSI APLIKASI DAN VISUALISASI HEATMAP
# =====================================================================

def predict_honey_adulteration_with_vis(input_image):
    if input_image is None:
        return "Silakan unggah citra spektrum terlebih dahulu.", "0.00%", "N/A", None, None

    # --- Tahap Preprocessing ---
    # 1. Simulasi pemotongan wilayah (Crop) area Karbohidrat (3-6 ppm)
    h, w, _ = input_image.shape
    crop_img = input_image[int(h*0.3):int(h*0.6), :]

    # 2. Resize gambar menjadi format input model (224, 224, 3)
    resized_img = cv2.resize(crop_img, (224, 224))

    # FIX: Membersihkan typo penulisan expand dims
    img_array = np.expand_dims(resized_img, axis=0)
    img_array = img_array.astype(np.float32) / 255.0 # Normalisasi nilai piksel

    # --- Tahap Forward Inference ---
    # 3. Ekstraksi fitur visual via CNN-CBAM
    feat = feature_extractor.predict(img_array)

    # 4. Standardisasi vektor fitur 1D
    scaled_feat = scaler.transform(feat)

    # 5. Klasifikasi biner menggunakan SVM
    prediction = svm_classifier.predict(scaled_feat)[0]
    probabilities = svm_classifier.predict_proba(scaled_feat)[0]

    # --- Pembuatan Ilustrasi Heatmap Atensi (SAM) ---
    gray_resized = cv2.cvtColor(resized_img, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray_resized, 50, 255, cv2.THRESH_BINARY_INV)
    heatmap = cv2.applyColorMap(thresh, cv2.COLORMAP_JET)

    # Menggabungkan gambar asli dengan heatmap (Overlay) sebagai ilustrasi SAM
    attention_visual = cv2.addWeighted(resized_img, 0.6, heatmap, 0.4, 0)

    # --- Pemetaan Label Output ---
    if prediction == 1:
        status = "MADU WOLFBERRY MURNI (Genuine Pure Honey)"
        confidence = probabilities[1] * 100
        color_alert = "Madu terdeteksi memiliki profil metabolit alami sesuai standar tanpa campuran sirup eksternal."
    else:
        status = "MADU TERADULTERASI / PALSU (Adulterated Honey)"
        confidence = probabilities[0] * 100
        color_alert = "Peringatan! Spektrum mengindikasikan adanya kandungan sirup gula tambahan (Jagung/Maltosa)."

    return status, f"{confidence:.2f}%", color_alert, resized_img, attention_visual

# =====================================================================
# 4. MEMBANGUN INTERFACE GUI MENGGUNAKAN GRADIO
# =====================================================================

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown(
        """
        # 🍯 Aplikasi Computer Vision: Autentikasi Madu Wolfberry
        ### Pendeteksian Pemalsuan Sirup Gula Berbasis Integrasi Model Hybrid CNN-CBAM-SVM & 1H NMR Spektroskopi
        *Aplikasi ini memproses citra mentah spektrum 1H NMR secara end-to-end, melakukan pemotongan otomatis pada wilayah karbohidrat (3-6 ppm), ekstraksi fitur berbasis atensi spatial-channel, dan klasifikasi menggunakan pembatas margin maksimal SVM.*
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📸 Input Citra Spektrum 1H NMR")
            input_img = gr.Image(type="numpy", label="Unggah Gambar Grafik Spektrum (.png / .jpg)")
            btn_submit = gr.Button("Analisis Keaslian Produk", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 📊 Hasil Analisis Laboratorium")
            output_status = gr.Textbox(label="Status Keputusan Akhir Model", interactive=False)
            output_conf = gr.Textbox(label="Tingkat Keyakinan (Confidence Score)", interactive=False)
            output_desc = gr.Markdown("### **Deskripsi Indikasi:**\nHasil analisis akan muncul setelah tombol ditekan.")

    # Baris visualisasi ilustrasi gambar hasil pemrosesan internal model
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🔍 1. Hasil Preprocessing (Crop Wilayah Karbohidrat 3-6 ppm)")
            out_crop = gr.Image(label="Citra Terpotong (224x224)", interactive=False)
        with gr.Column():
            gr.Markdown("### 🗺️ 2. Ilustrasi Atensi Spasial Model (CBAM SAM Heatmap)")
            out_heatmap = gr.Image(label="Overlay Heatmap Fitur Diskriminatif", interactive=False)

    # Menghubungkan interaksi tombol
    btn_submit.click(
        fn=predict_honey_adulteration_with_vis,
        inputs=[input_img],
        outputs=[output_status, output_conf, output_desc, out_crop, out_heatmap]
    )

    gr.Markdown(
        """
        ---
        **Catatan Evaluasi Akademik:** Aplikasi ini sukses mengimplementasikan ekstraksi fitur spasial/saluran dari artikel *Jia et al. (2025)* menggunakan Framework TensorFlow & Scikit-Learn untuk mengatasi kendala *Small-Sample Dataset* di industri pangan.
        """
    )

# Jalankan aplikasi secara langsung di dalam Google Colab
app.launch(debug=True)

Menginisialisasi simulasi pelatihan model hybrid CNN-CBAM-SVM...
2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 4s/step
Model hybrid CNN-CBAM-SVM berhasil dibuat dan siap digunakan!



/tmp/ipykernel_699/3623276613.py:165: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://afe624b501ebfe73b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://afe624b501ebfe73b7.gradio.live
